In [3]:
# ============================================================
# PROYECTO MUJERES STEM
# SNIES - MATRÍCULADOS
# UNIFICACIÓN Y HOMOLOGACIÓN 2014-2025
#
# CAPA: BRONZE → PLATA
#
# VARIABLES INSTITUCIONALES CONSERVADAS DESDE LAS FUENTES:
#   1. codigo_institucion
#   2. institucion
#   3. ies_padre
# ============================================================


# ============================================================
# PROCESO 1. CONECTAR GOOGLE DRIVE Y DEFINIR RUTAS
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import glob
import re
import unicodedata
import pandas as pd


BASE_PROYECTO = (
    "/content/drive/MyDrive/"
    "Proyecto Mujeres Stem"
)

CARPETA_BRONZE = (
    f"{BASE_PROYECTO}/Data/Bronze/SNIES/Matriculados"
)

CARPETA_PLATA = (
    f"{BASE_PROYECTO}/Data/Plata/Matriculados"
)

os.makedirs(CARPETA_PLATA, exist_ok=True)

print("=" * 70)
print("RUTAS DEL PROYECTO")
print("=" * 70)

print("Proyecto:", BASE_PROYECTO)
print("Bronze:", CARPETA_BRONZE)
print("Plata:", CARPETA_PLATA)


# ============================================================
# PROCESO 2. IDENTIFICAR ARCHIVOS DE MATRÍCULADOS
# ============================================================

archivos = [
    archivo
    for archivo in glob.glob(
        os.path.join(CARPETA_BRONZE, "*.xlsx")
    )
    if not os.path.basename(archivo).startswith("~$")
]

archivos = sorted(archivos)

print("\n" + "=" * 70)
print("ARCHIVOS DE MATRÍCULADOS")
print("=" * 70)

print("Número de archivos:", len(archivos))

for i, archivo in enumerate(archivos, 1):
    print(
        f"{i:02d}. {os.path.basename(archivo)}"
    )


# ============================================================
# PROCESO 3. DIAGNÓSTICO GENERAL DE LOS ARCHIVOS
# ============================================================

diagnostico = []

for archivo in archivos:

    df = pd.read_excel(
        archivo,
        dtype=str
    )

    diagnostico.append({
        "archivo": os.path.basename(archivo),
        "filas": len(df),
        "columnas": len(df.columns),
        "codigo_institucion":
            "SI" if "Código de la Institución" in df.columns
            else "NO",
        "institucion":
            "SI"
            if "Institución de Educación Superior (IES)"
            in df.columns
            else "NO",
        "ies_padre":
            "SI" if "IES_PADRE" in df.columns
            else "NO"
    })


diagnostico_df = pd.DataFrame(
    diagnostico
)

display(diagnostico_df)


# ============================================================
# PROCESO 4. REVISAR COLUMNAS DE CADA ARCHIVO
# ============================================================

for archivo in archivos:

    df = pd.read_excel(
        archivo,
        dtype=str
    )

    print("\n" + "=" * 80)
    print(
        "ARCHIVO:",
        os.path.basename(archivo)
    )
    print(
        "FILAS:",
        f"{len(df):,}"
    )
    print(
        "COLUMNAS:",
        len(df.columns)
    )
    print("=" * 80)

    for i, columna in enumerate(
        df.columns,
        1
    ):
        print(
            f"{i:02d}. {columna}"
        )


# ============================================================
# PROCESO 5. DEFINIR ESTRUCTURA MAESTRA
# ============================================================

COLUMNAS_MAESTRAS = [

    # --------------------------------------------------------
    # IDENTIFICACIÓN DE LA IES
    # --------------------------------------------------------

    "codigo_institucion",
    "institucion",
    "ies_padre",
    "principal_seccional",
    "tipo_ies",

    # --------------------------------------------------------
    # SECTOR
    # --------------------------------------------------------

    "id_sector",
    "sector",

    # --------------------------------------------------------
    # CARÁCTER Y ACREDITACIÓN
    # --------------------------------------------------------

    "ies_acreditada",
    "id_caracter",
    "caracter",
    "programa_acreditado",

    # --------------------------------------------------------
    # UBICACIÓN DE LA IES
    # --------------------------------------------------------

    "codigo_departamento_ies",
    "departamento_ies",
    "codigo_municipio_ies",
    "municipio_ies",

    # --------------------------------------------------------
    # PROGRAMA
    # --------------------------------------------------------

    "codigo_snies",
    "programa",

    # --------------------------------------------------------
    # NIVEL
    # --------------------------------------------------------

    "id_nivel_academico",
    "nivel_academico",
    "id_nivel_formacion",
    "nivel_formacion",

    # --------------------------------------------------------
    # METODOLOGÍA Y MODALIDAD
    # --------------------------------------------------------

    "id_metodologia",
    "metodologia",
    "id_modalidad",
    "modalidad",

    # --------------------------------------------------------
    # ÁREA
    # --------------------------------------------------------

    "id_area",
    "area_conocimiento",

    # --------------------------------------------------------
    # NBC
    # --------------------------------------------------------

    "id_nucleo",
    "nbc",

    # --------------------------------------------------------
    # UBICACIÓN DEL PROGRAMA
    # --------------------------------------------------------

    "codigo_departamento_programa",
    "departamento_programa",
    "codigo_municipio_programa",
    "municipio_programa",

    # --------------------------------------------------------
    # SEXO Y TIEMPO
    # --------------------------------------------------------

    "id_sexo",
    "sexo",
    "anio",
    "semestre",

    # --------------------------------------------------------
    # VARIABLE OBJETIVO
    # --------------------------------------------------------

    "matriculados",

    # --------------------------------------------------------
    # TRAZABILIDAD
    # --------------------------------------------------------

    "archivo_origen"
]


# ============================================================
# PROCESO 6. NORMALIZAR NOMBRES DE COLUMNAS
# ============================================================

def normalizar_nombre_columna(nombre):

    nombre = str(
        nombre
    ).strip().upper()

    nombre = unicodedata.normalize(
        "NFKD",
        nombre
    )

    nombre = "".join(
        c
        for c in nombre
        if not unicodedata.combining(c)
    )

    nombre = re.sub(
        r"\s+",
        " ",
        nombre
    )

    return nombre


# ============================================================
# PROCESO 7. DEFINIR HOMOLOGACIÓN DE COLUMNAS
# ============================================================

HOMOLOGACION = {

    # --------------------------------------------------------
    # INSTITUCIÓN
    # --------------------------------------------------------

    "CODIGO DE LA INSTITUCION":
        "codigo_institucion",

    "IES_PADRE":
        "ies_padre",

    "INSTITUCION DE EDUCACION SUPERIOR (IES)":
        "institucion",

    "PRINCIPAL O SECCIONAL":
        "principal_seccional",

    "TIPO IES":
        "tipo_ies",

    # --------------------------------------------------------
    # SECTOR
    # --------------------------------------------------------

    "ID_SECTOR":
        "id_sector",

    "ID SECTOR IES":
        "id_sector",

    "SECTOR IES":
        "sector",

    # --------------------------------------------------------
    # CARÁCTER
    # --------------------------------------------------------

    "ID_CARACTER":
        "id_caracter",

    "ID CARACTER":
        "id_caracter",

    "ID CARACTER IES":
        "id_caracter",

    "CARACTER IES":
        "caracter",

    # --------------------------------------------------------
    # ACREDITACIÓN
    # --------------------------------------------------------

    "IES ACREDITADA":
        "ies_acreditada",

    "PROGRAMA ACREDITADO":
        "programa_acreditado",

    # --------------------------------------------------------
    # UBICACIÓN IES
    # --------------------------------------------------------

    "CODIGO DEL DEPARTAMENTO (IES)":
        "codigo_departamento_ies",

    "CODIGO DEL DEPARTAMENTO IES":
        "codigo_departamento_ies",

    "DEPARTAMENTO DE DOMICILIO DE LA IES":
        "departamento_ies",

    "CODIGO DEL MUNICIPIO (IES)":
        "codigo_municipio_ies",

    "CODIGO DEL MUNICIPIO IES":
        "codigo_municipio_ies",

    "MUNICIPIO DE DOMICILIO DE LA IES":
        "municipio_ies",

    # --------------------------------------------------------
    # PROGRAMA
    # --------------------------------------------------------

    "CODIGO SNIES DEL PROGRAMA":
        "codigo_snies",

    "PROGRAMA ACADEMICO":
        "programa",

    # --------------------------------------------------------
    # NIVEL
    # --------------------------------------------------------

    "ID NIVEL":
        "id_nivel_academico",

    "ID NIVEL ACADEMICO":
        "id_nivel_academico",

    "NIVEL ACADEMICO":
        "nivel_academico",

    "ID NIVEL FORMACION":
        "id_nivel_formacion",

    "ID NIVEL DE FORMACION":
        "id_nivel_formacion",

    "NIVEL DE FORMACION":
        "nivel_formacion",

    # --------------------------------------------------------
    # METODOLOGÍA
    # --------------------------------------------------------

    "ID METODOLOGIA":
        "id_metodologia",

    "METODOLOGIA":
        "metodologia",

    # --------------------------------------------------------
    # MODALIDAD
    # --------------------------------------------------------

    "ID MODALIDAD":
        "id_modalidad",

    "MODALIDAD":
        "modalidad",

    # --------------------------------------------------------
    # ÁREA
    # --------------------------------------------------------

    "ID AREA":
        "id_area",

    "ID AREA DE CONOCIMIENTO":
        "id_area",

    "AREA DE CONOCIMIENTO":
        "area_conocimiento",

    # --------------------------------------------------------
    # NBC
    # --------------------------------------------------------

    "ID NUCLEO":
        "id_nucleo",

    "ID_NUCLEO":
        "id_nucleo",

    "NUCLEO BASICO DEL CONOCIMIENTO (NBC)":
        "nbc",

    # --------------------------------------------------------
    # UBICACIÓN DEL PROGRAMA
    # --------------------------------------------------------

    "CODIGO DEL DEPARTAMENTO (PROGRAMA)":
        "codigo_departamento_programa",

    "DEPARTAMENTO DE OFERTA DEL PROGRAMA":
        "departamento_programa",

    "CODIGO DEL MUNICIPIO (PROGRAMA)":
        "codigo_municipio_programa",

    "MUNICIPIO DE OFERTA DEL PROGRAMA":
        "municipio_programa",

    # --------------------------------------------------------
    # SEXO
    # --------------------------------------------------------

    "ID GENERO":
        "id_sexo",

    "ID SEXO":
        "id_sexo",

    "GENERO":
        "sexo",

    "SEXO":
        "sexo",

    # --------------------------------------------------------
    # TIEMPO
    # --------------------------------------------------------

    "ANO":
        "anio",

    "AÑO":
        "anio",

    "SEMESTRE":
        "semestre",

    # --------------------------------------------------------
    # MATRÍCULADOS
    # --------------------------------------------------------

    "MATRICULADOS":
        "matriculados"
}


# ============================================================
# PROCESO 8. PROCESAR Y HOMOLOGAR CADA ARCHIVO
# ============================================================

def procesar_archivo(archivo):

    nombre_archivo = os.path.basename(
        archivo
    )

    print("\n" + "=" * 70)
    print(
        "PROCESANDO:",
        nombre_archivo
    )
    print("=" * 70)

    # --------------------------------------------------------
    # 8.1 LEER ARCHIVO
    # --------------------------------------------------------

    df = pd.read_excel(
        archivo,
        dtype=str
    )

    filas_originales = len(df)
    columnas_originales = len(df.columns)

    # --------------------------------------------------------
    # 8.2 NORMALIZAR NOMBRES
    # --------------------------------------------------------

    df.columns = [
        normalizar_nombre_columna(c)
        for c in df.columns
    ]

    # --------------------------------------------------------
    # 8.3 ELIMINAR VARIABLES CINE
    # --------------------------------------------------------

    columnas_cine = [
        c
        for c in df.columns
        if "CINE" in c
    ]

    if columnas_cine:

        df = df.drop(
            columns=columnas_cine
        )

    # --------------------------------------------------------
    # 8.4 HOMOLOGAR COLUMNAS
    # --------------------------------------------------------

    nuevas_columnas = {}

    for columna in df.columns:

        if columna in HOMOLOGACION:

            nuevas_columnas[
                columna
            ] = HOMOLOGACION[
                columna
            ]

        elif re.match(
            r"^MATRICULADOS\s*\d{4}$",
            columna
        ):

            nuevas_columnas[
                columna
            ] = "matriculados"

    df = df.rename(
        columns=nuevas_columnas
    )

    # --------------------------------------------------------
    # 8.5 VALIDAR VARIABLE MATRÍCULADOS
    # --------------------------------------------------------

    if "matriculados" not in df.columns:

        raise ValueError(
            f"No se encontró la variable "
            f"MATRICULADOS en "
            f"{nombre_archivo}\n\n"
            f"Columnas encontradas:\n"
            f"{list(df.columns)}"
        )

    # --------------------------------------------------------
    # 8.6 DIAGNÓSTICO DE VARIABLES INSTITUCIONALES
    # --------------------------------------------------------

    print("\nVARIABLES DE IDENTIFICACIÓN INSTITUCIONAL")

    variables_ies = [
        "codigo_institucion",
        "institucion",
        "ies_padre"
    ]

    for variable in variables_ies:

        if variable in df.columns:

            faltantes = (
                df[variable]
                .isna()
                .sum()
            )

            distintos = (
                df[variable]
                .nunique(
                    dropna=True
                )
            )

            print(
                f"✓ {variable}: "
                f"PRESENTE | "
                f"distintos: {distintos:,} | "
                f"faltantes: {faltantes:,}"
            )

        else:

            print(
                f"⚠ {variable}: "
                f"NO EXISTE EN EL ARCHIVO"
            )

    # --------------------------------------------------------
    # 8.7 DETECTAR COLUMNAS DUPLICADAS
    # --------------------------------------------------------

    duplicadas = (
        df.columns[
            df.columns.duplicated()
        ].tolist()
    )

    if duplicadas:

        raise ValueError(
            f"Columnas duplicadas después "
            f"de homologar en "
            f"{nombre_archivo}: "
            f"{duplicadas}"
        )

    # --------------------------------------------------------
    # 8.8 TRAZABILIDAD
    # --------------------------------------------------------

    df["archivo_origen"] = (
        nombre_archivo
    )

    # --------------------------------------------------------
    # 8.9 CONSERVAR COLUMNAS MAESTRAS
    # --------------------------------------------------------

    columnas_presentes = [
        c
        for c in COLUMNAS_MAESTRAS
        if c in df.columns
    ]

    df = df[
        columnas_presentes
    ]

    # --------------------------------------------------------
    # 8.10 INFORME
    # --------------------------------------------------------

    print("\nRESUMEN DEL ARCHIVO")

    print(
        "Filas:",
        f"{filas_originales:,}"
    )

    print(
        "Columnas originales:",
        columnas_originales
    )

    print(
        "Columnas CINE eliminadas:",
        len(columnas_cine)
    )

    print(
        "Columnas finales:",
        len(df.columns)
    )

    return df


# ============================================================
# PROCESO 9. UNIFICAR TODOS LOS ARCHIVOS
# ============================================================

bases = []

for archivo in archivos:

    df_temp = procesar_archivo(
        archivo
    )

    bases.append(
        df_temp
    )


matriculados_unificado = pd.concat(
    bases,
    ignore_index=True,
    sort=False
)


print("\n" + "=" * 70)
print("BASE UNIFICADA DE MATRÍCULADOS")
print("=" * 70)

print(
    "Número total de registros:",
    f"{len(matriculados_unificado):,}"
)

print(
    "Número total de columnas:",
    len(matriculados_unificado.columns)
)


# ============================================================
# PROCESO 10. VALIDACIÓN DE LA BASE UNIFICADA
# ============================================================

TOTAL_ESPERADO = 846018

print("=" * 70)
print("VALIDACIÓN DE MATRÍCULADOS UNIFICADOS")
print("=" * 70)


# ------------------------------------------------------------
# 10.1 TOTAL DE REGISTROS
# ------------------------------------------------------------

total_registros = (
    len(matriculados_unificado)
)

print("\nTOTAL DE REGISTROS")
print("-" * 40)

print(
    "Total obtenido:",
    f"{total_registros:,}"
)

print(
    "Total esperado:",
    f"{TOTAL_ESPERADO:,}"
)

if total_registros == TOTAL_ESPERADO:

    print("✓ CORRECTO")

else:

    print("⚠ REVISAR")


# ------------------------------------------------------------
# 10.2 AÑOS
# ------------------------------------------------------------

print("\nAÑOS")
print("-" * 40)

print(
    sorted(
        matriculados_unificado[
            "anio"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# 10.3 REGISTROS POR AÑO
# ------------------------------------------------------------

print("\nREGISTROS POR AÑO")
print("-" * 40)

registros_anio = (
    matriculados_unificado
    .groupby("anio")
    .size()
    .reset_index(
        name="registros"
    )
)

display(
    registros_anio
)


# ------------------------------------------------------------
# 10.4 VARIABLES CINE
# ------------------------------------------------------------

print("\nVARIABLES CINE")
print("-" * 40)

columnas_cine = [
    c
    for c in matriculados_unificado.columns
    if "CINE" in c.upper()
]

if len(columnas_cine) == 0:

    print(
        "✓ No existen variables CINE."
    )

else:

    print(
        "⚠ Se encontraron:",
        columnas_cine
    )


# ------------------------------------------------------------
# 10.5 VALORES NULOS
# ------------------------------------------------------------

print("\nVALORES NULOS")
print("-" * 40)

nulos = (
    matriculados_unificado
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    nulos[
        nulos > 0
    ].to_frame(
        "faltantes"
    )
)


# ------------------------------------------------------------
# 10.6 DUPLICADOS EXACTOS
# ------------------------------------------------------------

print("\nDUPLICADOS EXACTOS")
print("-" * 40)

print(
    "Filas duplicadas:",
    matriculados_unificado
    .duplicated()
    .sum()
)


# ============================================================
# PROCESO 11. DEFINIR LAS VARIABLES FINALES
# ============================================================

COLUMNAS_DEFINITIVAS = [

    # IDENTIFICACIÓN INSTITUCIONAL
    "codigo_institucion",
    "institucion",
    "ies_padre",

    # CARACTERÍSTICAS IES
    "sector",
    "caracter",

    # UBICACIÓN IES
    "departamento_ies",

    # PROGRAMA
    "codigo_snies",
    "programa",

    # NIVEL
    "nivel_academico",

    # ÁREA
    "id_area",
    "area_conocimiento",

    # NBC
    "id_nucleo",
    "nbc",

    # SEXO
    "sexo",

    # TIEMPO
    "anio",
    "semestre",

    # MATRÍCULADOS
    "matriculados"
]


# ------------------------------------------------------------
# VALIDAR QUE TODAS LAS VARIABLES EXISTAN
# ------------------------------------------------------------

faltantes_columnas = [
    c
    for c in COLUMNAS_DEFINITIVAS
    if c not in matriculados_unificado.columns
]

if faltantes_columnas:

    raise ValueError(
        "Faltan columnas definitivas: "
        f"{faltantes_columnas}"
    )


matriculados_unificado = (
    matriculados_unificado[
        COLUMNAS_DEFINITIVAS
    ].copy()
)


print("=" * 70)
print("VARIABLES DEFINITIVAS")
print("=" * 70)

print(
    "Número de registros:",
    f"{len(matriculados_unificado):,}"
)

print(
    "Número de variables:",
    len(matriculados_unificado.columns)
)

for i, columna in enumerate(
    matriculados_unificado.columns,
    1
):

    print(
        f"{i:02d}. {columna}"
    )


# ============================================================
# PROCESO 12. CONVERSIÓN DE TIPOS DE DATOS
# ============================================================

# ------------------------------------------------------------
# 12.1 CÓDIGO DE INSTITUCIÓN
# ------------------------------------------------------------

matriculados_unificado[
    "codigo_institucion"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "codigo_institucion"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 12.2 IES PADRE
# ------------------------------------------------------------

matriculados_unificado[
    "ies_padre"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "ies_padre"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 12.3 CÓDIGO SNIES
# ------------------------------------------------------------

snies_no_numericos = (
    matriculados_unificado
    .loc[
        ~matriculados_unificado[
            "codigo_snies"
        ]
        .astype("string")
        .str.fullmatch(
            r"\d+"
        ),
        "codigo_snies"
    ]
    .drop_duplicates()
)


print(
    "Valores SNIES no numéricos:",
    len(snies_no_numericos)
)

if len(snies_no_numericos) > 0:

    display(
        snies_no_numericos.head(50)
    )

    raise ValueError(
        "Existen códigos SNIES no numéricos."
    )


matriculados_unificado[
    "codigo_snies"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "codigo_snies"
        ],
        errors="raise"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 12.4 ID ÁREA
# ------------------------------------------------------------

matriculados_unificado[
    "id_area"
] = (
    matriculados_unificado[
        "id_area"
    ]
    .astype("string")
)


# ------------------------------------------------------------
# 12.5 AÑO
# ------------------------------------------------------------

matriculados_unificado[
    "anio"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "anio"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 12.6 SEMESTRE
# ------------------------------------------------------------

matriculados_unificado[
    "semestre"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "semestre"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 12.7 MATRÍCULADOS
# ------------------------------------------------------------

matriculados_unificado[
    "matriculados"
] = (
    pd.to_numeric(
        matriculados_unificado[
            "matriculados"
        ],
        errors="coerce"
    )
    .astype("Int64")
)


print("=" * 70)
print("TIPOS DE DATOS")
print("=" * 70)

display(
    matriculados_unificado
    .dtypes
    .to_frame("tipo")
)


# ============================================================
# PROCESO 13. HOMOLOGACIÓN DE VARIABLES CATEGÓRICAS
# ============================================================


# ------------------------------------------------------------
# 13.1 SECTOR
# ------------------------------------------------------------

matriculados_unificado[
    "sector"
] = (
    matriculados_unificado[
        "sector"
    ]
    .astype("string")
    .str.strip()
    .replace({
        "OFICIAL": "Oficial",
        "Oficial": "Oficial",
        "PRIVADA": "Privada",
        "Privado": "Privada"
    })
)


# ------------------------------------------------------------
# 13.2 CARÁCTER
# ------------------------------------------------------------

matriculados_unificado[
    "caracter"
] = (
    matriculados_unificado[
        "caracter"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({

        "UNIVERSIDAD":
            "Universidad",

        "INSTITUCIÓN UNIVERSITARIA/ESCUELA TECNOLÓGICA":
            "Institución Universitaria/Escuela Tecnológica",

        "INSTITUCION UNIVERSITARIA/ESCUELA TECNOLOGICA":
            "Institución Universitaria/Escuela Tecnológica",

        "INSTITUCIÓN TECNOLÓGICA":
            "Institución Tecnológica",

        "INSTITUCION TECNOLOGICA":
            "Institución Tecnológica",

        "INSTITUCIÓN TÉCNICA PROFESIONAL":
            "Institución Técnica Profesional",

        "INSTITUCION TECNICA PROFESIONAL":
            "Institución Técnica Profesional"
    })
)


# ------------------------------------------------------------
# 13.3 NIVEL ACADÉMICO
# ------------------------------------------------------------

matriculados_unificado[
    "nivel_academico"
] = (
    matriculados_unificado[
        "nivel_academico"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({
        "PREGRADO": "Pregrado",
        "POSGRADO": "Posgrado"
    })
)


# ------------------------------------------------------------
# 13.4 SEXO
# ------------------------------------------------------------

matriculados_unificado[
    "sexo"
] = (
    matriculados_unificado[
        "sexo"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({

        "FEMENINO": "Mujer",
        "MUJER": "Mujer",

        "MASCULINO": "Hombre",
        "HOMBRE": "Hombre",

        "NO BINARIO": "No binario",
        "TRANS": "Trans"
    })
)

print(
    "✓ Homologación categórica realizada."
)


# ============================================================
# PROCESO 14. HOMOLOGACIÓN DE DEPARTAMENTO DE LA IES
# ============================================================

def normalizar_texto(valor):

    if pd.isna(valor):

        return pd.NA

    valor = (
        str(valor)
        .strip()
        .upper()
    )

    valor = unicodedata.normalize(
        "NFKD",
        valor
    )

    valor = "".join(
        c
        for c in valor
        if not unicodedata.combining(c)
    )

    valor = re.sub(
        r"\s+",
        " ",
        valor
    )

    return valor


matriculados_unificado[
    "departamento_ies"
] = (
    matriculados_unificado[
        "departamento_ies"
    ]
    .apply(normalizar_texto)
)


MAPA_DEPARTAMENTOS = {

    "BOGOTA D.C":
        "BOGOTA D.C.",

    "BOGOTA D.C.":
        "BOGOTA D.C.",

    "NARINIO":
        "NARINO",

    "GUAJIRA":
        "LA GUAJIRA",

    "SAN ANDRES Y PROVIDENCIA":
        "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA",

    "SAN ANDRES Y PROVI":
        "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA",

    "ARCHIPIELAGO DE SA":
        "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA",

    "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA":
        "ARCHIPIELAGO DE SAN ANDRES, PROVIDENCIA Y SANTA CATALINA"
}


matriculados_unificado[
    "departamento_ies"
] = (
    matriculados_unificado[
        "departamento_ies"
    ]
    .replace(
        MAPA_DEPARTAMENTOS
    )
)


print("=" * 70)
print("DEPARTAMENTOS HOMOLOGADOS")
print("=" * 70)

display(
    matriculados_unificado[
        "departamento_ies"
    ]
    .value_counts(
        dropna=False
    )
    .to_frame(
        "frecuencia"
    )
)

print(
    "Número de categorías:",
    matriculados_unificado[
        "departamento_ies"
    ].nunique(
        dropna=True
    )
)


# ============================================================
# PROCESO 15. HOMOLOGACIÓN DE ÁREA DE CONOCIMIENTO
# ============================================================

area_normalizada = (
    matriculados_unificado[
        "area_conocimiento"
    ]
    .apply(normalizar_texto)
)


MAPA_AREA_CONOCIMIENTO = {

    "ECONOMIA ADMINISTRACION CONTADURIA Y AFINES":
        "Economía, administración, contaduría y afines",

    "ECONOMIA, ADMINISTRACION, CONTADURIA Y AFINES":
        "Economía, administración, contaduría y afines",

    "INGENIERIA ARQUITECTURA URBANISMO Y AFINES":
        "Ingeniería, arquitectura, urbanismo y afines",

    "INGENIERIA, ARQUITECTURA, URBANISMO Y AFINES":
        "Ingeniería, arquitectura, urbanismo y afines",

    "CIENCIAS DE LA EDUCACION":
        "Ciencias de la educación",

    "CIENCIAS DE LA SALUD":
        "Ciencias de la salud",

    "CIENCIAS SOCIALES Y HUMANAS":
        "Ciencias sociales y humanas",

    "BELLAS ARTES":
        "Bellas artes",

    "AGRONOMIA VETERINARIA Y AFINES":
        "Agronomía, veterinaria y afines",

    "AGRONOMIA, VETERINARIA Y AFINES":
        "Agronomía, veterinaria y afines",

    "MATEMATICAS Y CIENCIAS NATURALES":
        "Matemáticas y ciencias naturales",

    "SIN INFORMACION":
        "Sin información",

    "SIN CLASIFICAR":
        "Sin clasificar"
}


matriculados_unificado[
    "area_conocimiento"
] = (
    area_normalizada
    .replace(
        MAPA_AREA_CONOCIMIENTO
    )
)


resultado_area = (
    matriculados_unificado[
        "area_conocimiento"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "area_conocimiento"
    )
    .reset_index(
        name="frecuencia"
    )
)

display(
    resultado_area
)

print(
    "Categorías:",
    matriculados_unificado[
        "area_conocimiento"
    ].nunique(
        dropna=True
    )
)


# ============================================================
# PROCESO 16. HOMOLOGACIÓN DE NBC
# ============================================================

nbc_diagnostico = (
    matriculados_unificado[
        "nbc"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "valor_original"
    )
    .reset_index(
        name="frecuencia"
    )
)


nbc_diagnostico[
    "valor_normalizado"
] = (
    nbc_diagnostico[
        "valor_original"
    ]
    .apply(normalizar_texto)
)


MAPA_NBC = dict(
    zip(
        nbc_diagnostico[
            "valor_original"
        ],
        nbc_diagnostico[
            "valor_normalizado"
        ]
    )
)


matriculados_unificado[
    "nbc_normalizado"
] = (
    matriculados_unificado[
        "nbc"
    ]
    .map(
        MAPA_NBC
    )
)


# ------------------------------------------------------------
# VALIDAR RELACIÓN ID_NUCLEO - NBC
# ------------------------------------------------------------

relacion_nucleo_nbc = (
    matriculados_unificado
    .groupby(
        "id_nucleo",
        dropna=False
    )[
        "nbc_normalizado"
    ]
    .nunique(
        dropna=True
    )
    .reset_index(
        name="numero_nbc"
    )
)


casos_multiples_nbc = (
    relacion_nucleo_nbc[
        relacion_nucleo_nbc[
            "numero_nbc"
        ] > 1
    ]
)

print(
    "ID_NUCLEO con más de un NBC:",
    len(casos_multiples_nbc)
)


# ------------------------------------------------------------
# APLICAR NBC NORMALIZADO
# ------------------------------------------------------------

matriculados_unificado[
    "nbc"
] = (
    matriculados_unificado[
        "nbc_normalizado"
    ]
)


matriculados_unificado.drop(
    columns=[
        "nbc_normalizado"
    ],
    inplace=True
)


print(
    "Categorías NBC:",
    matriculados_unificado[
        "nbc"
    ].nunique(
        dropna=True
    )
)


# ============================================================
# PROCESO 17. VALIDACIÓN DE IDENTIFICACIÓN INSTITUCIONAL
# ============================================================

print("=" * 70)
print("VALIDACIÓN DE IDENTIFICACIÓN INSTITUCIONAL")
print("=" * 70)


# ------------------------------------------------------------
# 17.1 CÓDIGOS DE INSTITUCIÓN
# ------------------------------------------------------------

print(
    "\nCódigos de institución:",
    matriculados_unificado[
        "codigo_institucion"
    ].nunique(
        dropna=True
    )
)


# ------------------------------------------------------------
# 17.2 INSTITUCIONES
# ------------------------------------------------------------

print(
    "Instituciones:",
    matriculados_unificado[
        "institucion"
    ].nunique(
        dropna=True
    )
)


# ------------------------------------------------------------
# 17.3 IES PADRE
# ------------------------------------------------------------

print(
    "IES padre:",
    matriculados_unificado[
        "ies_padre"
    ].nunique(
        dropna=True
    )
)


# ------------------------------------------------------------
# 17.4 FALTANTES
# ------------------------------------------------------------

print("\nFALTANTES")
print("-" * 50)

faltantes_ies = (
    matriculados_unificado[
        [
            "codigo_institucion",
            "institucion",
            "ies_padre"
        ]
    ]
    .isna()
    .sum()
)

display(
    faltantes_ies.to_frame(
        "cantidad_faltantes"
    )
)


# ------------------------------------------------------------
# 17.5 CÓDIGO → INSTITUCIÓN
# ------------------------------------------------------------

relacion_codigo_institucion = (
    matriculados_unificado
    .groupby(
        "codigo_institucion",
        dropna=False
    )[
        "institucion"
    ]
    .nunique(
        dropna=True
    )
)


inconsistencias_institucion = (
    relacion_codigo_institucion > 1
).sum()


print(
    "\nCódigos con más de un nombre "
    "de institución:",
    inconsistencias_institucion
)


# ------------------------------------------------------------
# 17.6 CÓDIGO → IES PADRE
# ------------------------------------------------------------

relacion_codigo_padre = (
    matriculados_unificado
    .groupby(
        "codigo_institucion",
        dropna=False
    )[
        "ies_padre"
    ]
    .nunique(
        dropna=True
    )
)


inconsistencias_padre = (
    relacion_codigo_padre > 1
).sum()


print(
    "Códigos con más de un IES_PADRE:",
    inconsistencias_padre
)


# ============================================================
# PROCESO 18. NORMALIZACIÓN Y DEPURACIÓN DE PROGRAMAS
# ============================================================

def normalizar_programa(texto):

    if pd.isna(texto):

        return pd.NA

    texto = (
        str(texto)
        .strip()
        .upper()
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        c
        for c in texto
        if not unicodedata.combining(c)
    )

    return texto


matriculados_unificado[
    "programa_normalizado"
] = (
    matriculados_unificado[
        "programa"
    ]
    .apply(
        normalizar_programa
    )
)


# ------------------------------------------------------------
# RELACIÓN SNIES - PROGRAMA
# ------------------------------------------------------------

programas_por_snies_normalizados = (
    matriculados_unificado
    .groupby(
        "codigo_snies"
    )[
        "programa_normalizado"
    ]
    .nunique()
)


print(
    "SNIES con un solo nombre:",
    (
        programas_por_snies_normalizados
        == 1
    ).sum()
)


print(
    "SNIES con múltiples nombres:",
    (
        programas_por_snies_normalizados
        > 1
    ).sum()
)


print(
    "Máximo de nombres por SNIES:",
    programas_por_snies_normalizados.max()
)


# ------------------------------------------------------------
# IDENTIFICAR SNIES A EXCLUIR
# ------------------------------------------------------------

snies_excluir = (
    programas_por_snies_normalizados[
        programas_por_snies_normalizados
        > 1
    ]
    .index
)


print(
    "\nSNIES a excluir:",
    len(snies_excluir)
)


# ------------------------------------------------------------
# ELIMINAR REGISTROS PROBLEMÁTICOS
# ------------------------------------------------------------

matriculados_unificado = (
    matriculados_unificado[
        ~matriculados_unificado[
            "codigo_snies"
        ]
        .isin(
            snies_excluir
        )
    ]
    .copy()
)


# ------------------------------------------------------------
# UTILIZAR NOMBRE NORMALIZADO
# ------------------------------------------------------------

matriculados_unificado[
    "programa"
] = (
    matriculados_unificado[
        "programa_normalizado"
    ]
)


matriculados_unificado.drop(
    columns=[
        "programa_normalizado"
    ],
    inplace=True
)


print("\nRESULTADO")
print("-" * 40)

print(
    "Registros:",
    f"{len(matriculados_unificado):,}"
)

print(
    "Códigos SNIES:",
    matriculados_unificado[
        "codigo_snies"
    ].nunique()
)

print(
    "Programas:",
    matriculados_unificado[
        "programa"
    ].nunique()
)

print(
    "Faltantes programa:",
    matriculados_unificado[
        "programa"
    ].isna().sum()
)


# ============================================================
# PROCESO 19. VALIDACIÓN DE LA VARIABLE MATRÍCULADOS
# ============================================================

print("=" * 70)
print("VALIDACIÓN DE MATRÍCULADOS")
print("=" * 70)

print(
    "Tipo:",
    matriculados_unificado[
        "matriculados"
    ].dtype
)

print(
    "Mínimo:",
    matriculados_unificado[
        "matriculados"
    ].min()
)

print(
    "Máximo:",
    matriculados_unificado[
        "matriculados"
    ].max()
)

print(
    "Suma total:",
    matriculados_unificado[
        "matriculados"
    ].sum()
)

print(
    "Ceros:",
    (
        matriculados_unificado[
            "matriculados"
        ] == 0
    ).sum()
)

print(
    "Negativos:",
    (
        matriculados_unificado[
            "matriculados"
        ] < 0
    ).sum()
)

print(
    "Faltantes:",
    matriculados_unificado[
        "matriculados"
    ].isna().sum()
)

display(
    matriculados_unificado[
        "matriculados"
    ].describe()
)


# ============================================================
# PROCESO 20. VALIDACIÓN FINAL DE LA BASE MAESTRA
# ============================================================

print("=" * 70)
print("VALIDACIÓN FINAL — MATRÍCULADOS")
print("=" * 70)


# ------------------------------------------------------------
# 20.1 ESTRUCTURA
# ------------------------------------------------------------

print("\nESTRUCTURA")
print("-" * 40)

print(
    "Registros:",
    f"{len(matriculados_unificado):,}"
)

print(
    "Variables:",
    len(matriculados_unificado.columns)
)


# ------------------------------------------------------------
# 20.2 COLUMNAS
# ------------------------------------------------------------

print("\nCOLUMNAS")
print("-" * 40)

for i, columna in enumerate(
    matriculados_unificado.columns,
    1
):

    print(
        f"{i:02d}. {columna}"
    )


# ------------------------------------------------------------
# 20.3 AÑOS
# ------------------------------------------------------------

print("\nAÑOS")
print("-" * 40)

print(
    sorted(
        matriculados_unificado[
            "anio"
        ]
        .dropna()
        .unique()
        .tolist()
    )
)


# ------------------------------------------------------------
# 20.4 REGISTROS POR AÑO
# ------------------------------------------------------------

print("\nREGISTROS POR AÑO")
print("-" * 40)

display(
    matriculados_unificado
    .groupby("anio")
    .size()
    .sort_index()
)


# ------------------------------------------------------------
# 20.5 FALTANTES
# ------------------------------------------------------------

print("\nVALORES FALTANTES")
print("-" * 40)

faltantes_final = (
    matriculados_unificado
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

display(
    faltantes_final[
        faltantes_final > 0
    ]
    .to_frame(
        "cantidad_faltantes"
    )
)


# ------------------------------------------------------------
# 20.6 DUPLICADOS EXACTOS
# ------------------------------------------------------------

print("\nDUPLICADOS EXACTOS")
print("-" * 40)

print(
    "Duplicados:",
    matriculados_unificado
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# 20.7 TIPOS
# ------------------------------------------------------------

print("\nTIPOS DE DATOS")
print("-" * 40)

display(
    matriculados_unificado
    .dtypes
    .to_frame(
        "tipo"
    )
)


# ============================================================
# PROCESO 21. VALIDACIÓN ESPECÍFICA DE LAS TRES VARIABLES IES
# ============================================================

print("=" * 70)
print("VALIDACIÓN FINAL DE LAS VARIABLES INSTITUCIONALES")
print("=" * 70)

resumen_ies = pd.DataFrame({

    "variable": [
        "codigo_institucion",
        "institucion",
        "ies_padre"
    ],

    "registros": [
        len(matriculados_unificado),
        len(matriculados_unificado),
        len(matriculados_unificado)
    ],

    "con_dato": [
        matriculados_unificado[
            "codigo_institucion"
        ].notna().sum(),

        matriculados_unificado[
            "institucion"
        ].notna().sum(),

        matriculados_unificado[
            "ies_padre"
        ].notna().sum()
    ],

    "faltantes": [
        matriculados_unificado[
            "codigo_institucion"
        ].isna().sum(),

        matriculados_unificado[
            "institucion"
        ].isna().sum(),

        matriculados_unificado[
            "ies_padre"
        ].isna().sum()
    ],

    "valores_distintos": [
        matriculados_unificado[
            "codigo_institucion"
        ].nunique(dropna=True),

        matriculados_unificado[
            "institucion"
        ].nunique(dropna=True),

        matriculados_unificado[
            "ies_padre"
        ].nunique(dropna=True)
    ]
})

display(
    resumen_ies
)


# ============================================================
# PROCESO 22. GUARDAR BASE MAESTRA EN PLATA
# ============================================================

ruta_csv = os.path.join(
    CARPETA_PLATA,
    "SNIES_Matriculados_Unificado_2014_2025.csv"
)

ruta_parquet = os.path.join(
    CARPETA_PLATA,
    "SNIES_Matriculados_Unificado_2014_2025.parquet"
)


# ------------------------------------------------------------
# GUARDAR CSV
# ------------------------------------------------------------

matriculados_unificado.to_csv(
    ruta_csv,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# GUARDAR PARQUET
# ------------------------------------------------------------

matriculados_unificado.to_parquet(
    ruta_parquet,
    index=False
)


print("=" * 70)
print("BASE MAESTRA DE MATRÍCULADOS GUARDADA")
print("=" * 70)

print(
    "Registros:",
    f"{len(matriculados_unificado):,}"
)

print(
    "Variables:",
    len(matriculados_unificado.columns)
)

print(
    "Periodo:",
    matriculados_unificado[
        "anio"
    ].min(),
    "–",
    matriculados_unificado[
        "anio"
    ].max()
)

print("\nCSV:")
print(ruta_csv)

print("\nParquet:")
print(ruta_parquet)


# ============================================================
# PROCESO 23. VERIFICAR ARCHIVOS DE SALIDA
# ============================================================

print("=" * 70)
print("VERIFICACIÓN DE ARCHIVOS")
print("=" * 70)

for archivo in [
    ruta_csv,
    ruta_parquet
]:

    if os.path.exists(archivo):

        tamano_mb = (
            os.path.getsize(archivo)
            / (1024 ** 2)
        )

        print(
            f"✓ {os.path.basename(archivo)}"
        )

        print(
            f"  Tamaño: {tamano_mb:.2f} MB"
        )

    else:

        print(
            f"✗ NO ENCONTRADO: {archivo}"
        )


print(
    "\nContenido de Plata/Matriculados:"
)

print(
    os.listdir(
        CARPETA_PLATA
    )
)

Mounted at /content/drive
RUTAS DEL PROYECTO
Proyecto: /content/drive/MyDrive/Proyecto Mujeres Stem
Bronze: /content/drive/MyDrive/Proyecto Mujeres Stem/Data/Bronze/SNIES/Matriculados
Plata: /content/drive/MyDrive/Proyecto Mujeres Stem/Data/Plata/Matriculados

ARCHIVOS DE MATRÍCULADOS
Número de archivos: 12
01. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2014.xlsx
02. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2015.xlsx
03. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2018.xlsx
04. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2019.xlsx
05. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2020.xlsx
06. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2021.xlsx
07. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2022.xlsx
08. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2023.xlsx
09. ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2024.xlsx
10. MATRICULADOS EN EDUCACIÓN SUP

,archivo,filas,columnas,codigo_institucion,institucion,ies_padre
0,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,60838,33,NO,SI,NO
1,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,59996,33,NO,SI,NO
2,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,63385,33,NO,SI,NO
3,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,70232,39,NO,NO,SI
4,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,69585,39,NO,NO,SI
5,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,70607,41,NO,NO,SI
6,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,69755,41,NO,NO,SI
7,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,72373,41,NO,NO,NO
8,ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIO...,75175,41,NO,NO,NO
9,MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA...,60363,33,NO,SI,NO



ARCHIVO: ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2014.xlsx
FILAS: 60,838
COLUMNAS: 33
01. Código de 
la Institución
02. IES PADRE
03. Institución de Educación Superior (IES)
04. Principal
 o
Seccional
05. Id_Sector
06. Sector IES
07. Id_Caracter
08. Caracter IES
09. Código del 
departamento
(IES)
10. Departamento de 
domicilio de la IES
11. Código del 
Municipio
(IES)
12. Municipio de
domicilio de la IES
13. Código 
SNIES del
programa
14. Programa Académico
15. Id_Nivel
16. Nivel Académico
17. Id_Nivel_Formacion
18. Nivel de Formación
19. Id_Metodologia
20. Metodología
21. Id_Area
22. Área de Conocimiento
23. Id_Nucleo
24. Núcleo Básico del Conocimiento (NBC)
25. Código del 
Departamento
(Programa)
26. Departamento de oferta del programa
27. Código del 
Municipio
(Programa)
28. Municipio de oferta del programa
29. Id Género
30. Género
31. Año
32. Semestre
33. Matriculados 2014

ARCHIVO: ESTUDIANTES MATRICULADOS EN EDUCACIÓN SUPERIOR - COLOMBIA 2015.xlsx
FILAS: 59,9

,anio,registros
0,2014,60838
1,2015,59996
2,2016,60363
3,2017,65382
4,2018,63385
5,2019,70232
6,2020,69585
7,2021,70607
8,2022,69755
9,2023,72373



VARIABLES CINE
----------------------------------------
✓ No existen variables CINE.

VALORES NULOS
----------------------------------------


...


DUPLICADOS EXACTOS
----------------------------------------
Duplicados: 139120

TIPOS DE DATOS
----------------------------------------


,tipo
codigo_institucion,Int64
institucion,object
ies_padre,Int64
sector,string[python]
caracter,string[python]
departamento_ies,object
codigo_snies,Int64
programa,object
nivel_academico,string[python]
id_area,string[python]


VALIDACIÓN FINAL DE LAS VARIABLES INSTITUCIONALES


,variable,registros,con_dato,faltantes,valores_distintos
0,codigo_institucion,790637,790637,0,347
1,institucion,790637,790637,0,404
2,ies_padre,790637,261324,529313,284


BASE MAESTRA DE MATRÍCULADOS GUARDADA
Registros: 790,637
Variables: 17
Periodo: 2014 – 2025

CSV:
/content/drive/MyDrive/Proyecto Mujeres Stem/Data/Plata/Matriculados/SNIES_Matriculados_Unificado_2014_2025.csv

Parquet:
/content/drive/MyDrive/Proyecto Mujeres Stem/Data/Plata/Matriculados/SNIES_Matriculados_Unificado_2014_2025.parquet
VERIFICACIÓN DE ARCHIVOS
✓ SNIES_Matriculados_Unificado_2014_2025.csv
  Tamaño: 165.75 MB
✓ SNIES_Matriculados_Unificado_2014_2025.parquet
  Tamaño: 3.87 MB

Contenido de Plata/Matriculados:
['SNIES_Matriculados_Unificado_2014_2025.csv', 'SNIES_Matriculados_Unificado_2014_2025.parquet']
